# 7. 가중치 반영 → 다익스트라 경로 탐색 + folium 시각화

## 1) import & 데이터 로드

In [1]:
from pathlib import Path
import pickle
import math
import pandas as pd
import networkx as nx

In [2]:
ARTIFACTS_DIR = Path("../artifacts")

# 그래프 로드 (x,y / lat,lng / length_m 포함)
with open(ARTIFACTS_DIR / "graph_xy.pkl", "rb") as f:
    G = pickle.load(f)

In [3]:
# (선택) 6단계에서 저장했다면, 간선별 그늘 길이 테이블 병합
# - 둘 중 하나만 있으면 자동으로 사용
edges_shadow_path = ARTIFACTS_DIR / "edges_with_shadow.parquet"
if edges_shadow_path.exists():
    df_edges = pd.read_parquet(edges_shadow_path)
    # 무향 그래프라 (u,v) / (v,u) 모두 매칭되도록 세팅
    shadow_lookup = {}
    for r in df_edges.itertuples(index=False):
        u, v = int(r.u), int(r.v)
        shadow_lookup[(u, v)] = float(r.shadow_len)
        shadow_lookup[(v, u)] = float(r.shadow_len)

    missing = 0
    for u, v, d in G.edges(data=True):
        shaded = shadow_lookup.get((u, v))
        if shaded is None:
            missing += 1
            shaded = 0.0
        d["shaded_len_m"] = float(shaded)
    print(f"[merge] 간선 그늘 병합 완료. 매칭 실패 {missing}개.")
else:
    # 5단계 대안 경로에서 이미 G에 'shaded_len_m'이 있을 수도 있음
    if not all(("shaded_len_m" in d) for _,_,d in G.edges(data=True)):
        raise RuntimeError("그늘 길이(shaded_len_m)를 찾을 수 없습니다. Step 5/6를 먼저 완료하세요.")

[merge] 간선 그늘 병합 완료. 매칭 실패 0개.


## 2) 가중치 → 간선 비용 계산 함수

* 간선 최종 가중치 = (기본거리×거리가중치 − 그늘거리×그늘가중치) / 10
* 제약: 거리 w∈[5,10], 그늘 w∈[0,5], 합=10

In [4]:
def apply_edge_costs(G: nx.Graph, w_dist: float, w_shade: float):
    """
    (w_dist + w_shade = 10), w_dist∈[5,10], w_shade∈[0,5]
    각 간선에 cost 속성을 채움.
    """
    # 검증/보정
    w_dist = float(w_dist); w_shade = float(w_shade)
    if abs((w_dist + w_shade) - 10.0) > 1e-9:
        raise ValueError("w_dist + w_shade는 10이어야 합니다.")
    if not (5.0 <= w_dist <= 10.0):
        raise ValueError("w_dist는 [5,10] 범위여야 합니다.")
    if not (0.0 <= w_shade <= 5.0):
        raise ValueError("w_shade는 [0,5] 범위여야 합니다.")

    for u, v, d in G.edges(data=True):
        base_len = float(d.get("length_m", d.get("length_xy_m", 0.0)))  # 지오데식 길이 우선
        shade_len = float(d.get("shaded_len_m", 0.0))
        # 음수 방지(전 구간 그늘일 때 값이 너무 작아지는 등 안전장치)
        cost = (w_dist * base_len - w_shade * shade_len) / 10.0
        d["cost"] = max(cost, 1e-9)  # 다익스트라 안정성


## 3) 경로 탐색 유틸

In [5]:
def summarize_path(G: nx.Graph, path: list[int]) -> dict:
    total_len = 0.0
    total_shade = 0.0
    total_cost = 0.0
    for a, b in zip(path[:-1], path[1:]):
        d = G[a][b]
        total_len  += float(d.get("length_m", d.get("length_xy_m", 0.0)))
        total_shade+= float(d.get("shaded_len_m", 0.0))
        total_cost += float(d.get("cost", 0.0))
    ratio = (total_shade / total_len) if total_len > 0 else 0.0
    return {
        "n_edges": len(path)-1,
        "total_len_m": total_len,
        "total_shade_m": total_shade,
        "shade_ratio": ratio,
        "total_cost": total_cost
    }

def find_route(G: nx.Graph, src: int, dst: int, w_dist=7.0, w_shade=3.0):
    """
    w_dist + w_shade = 10 가정.
    """
    apply_edge_costs(G, w_dist, w_shade)
    path = nx.shortest_path(G, source=src, target=dst, weight="cost")
    info = summarize_path(G, path)
    return path, info


## 4) 예시: 임의 두 노드로 경로 찾기

In [6]:
# 예시 시작/도착 노드 (그래프에 존재하는 id로 설정)
nodes_list = list(G.nodes)
src = nodes_list[0]
dst = nodes_list[-1]

# 가중치 예시 (합=10)
w_dist = 8.0
w_shade = 2.0

path, info = find_route(G, src, dst, w_dist=w_dist, w_shade=w_shade)
print("경로 노드 수:", len(path))
print("요약:", info)
print("경로 노드:", path[:10], "...", path[-10:])  # 너무 길면 일부만 표시


경로 노드 수: 20
요약: {'n_edges': 19, 'total_len_m': 909.1679547812332, 'total_shade_m': 33.9119007728808, 'shade_ratio': 0.03729992967145524, 'total_cost': 720.5519836704106}
경로 노드: [1, 113, 111, 112, 232, 120, 304, 158, 303, 70] ... [302, 133, 131, 206, 132, 238, 9, 36, 329, 330]


## 5) folium 시각화 (경로 오버레이)

In [7]:
import folium
from statistics import mean

In [8]:
# 그래프 중심으로 초기 맵 생성
lat_center = mean([G.nodes[n]["lat"] for n in G.nodes])
lng_center = mean([G.nodes[n]["lng"] for n in G.nodes])
m = folium.Map(location=[lat_center, lng_center], zoom_start=16, tiles="cartodbpositron")

In [9]:
# 1) 전체 간선(연한 파란색)
for u, v in G.edges:
    latlngs = [
        (G.nodes[u]["lat"], G.nodes[u]["lng"]),
        (G.nodes[v]["lat"], G.nodes[v]["lng"])
    ]
    folium.PolyLine(latlngs, weight=2, opacity=0.4, color="#3388ff").add_to(m)

# 2) 최단 경로(굵고 선명하게)
path_latlngs = [(G.nodes[n]["lat"], G.nodes[n]["lng"]) for n in path]
folium.PolyLine(path_latlngs, weight=6, color="#ff4d4f", opacity=0.9).add_to(m)

# 3) 출발/도착 마커
folium.CircleMarker(path_latlngs[0], radius=6, color="#2ecc71", fill=True, fill_opacity=1.0, popup=f"START {src}").add_to(m)
folium.CircleMarker(path_latlngs[-1], radius=6, color="#e74c3c", fill=True, fill_opacity=1.0, popup=f"END {dst}").add_to(m)

# 4) 경로 요약 툴팁
tooltip = (f"총거리: {info['total_len_m']:.1f} m<br>"
           f"그늘길이: {info['total_shade_m']:.1f} m "
           f"({info['shade_ratio']*100:.1f}%)<br>"
           f"총비용(cost): {info['total_cost']:.1f}<br>"
           f"w_dist={w_dist}, w_shade={w_shade}")
folium.Marker(path_latlngs[len(path_latlngs)//2], tooltip=tooltip).add_to(m)

html로 저장

In [10]:
m.save("../artifacts/route_map.html")  # folium Map 객체 m

## 6) 그림자도 올리기

In [11]:
# === 셀 0: ShadowRect 정의 (언피클보다 먼저 실행!) ===
from dataclasses import dataclass
from typing import List, Dict, Any, Tuple
import math

@dataclass
class ShadowRect:
    cx: float      # 직사각형 중심 x  (center = c + (L/2)·d)
    cy: float      # 직사각형 중심 y
    length: float  # L
    width: float   # 2r
    dx: float      # d_x (태양 반대 단위벡터)
    dy: float      # d_y
    meta: Dict[str, Any]

    def endpoints(self) -> Tuple[Tuple[float,float], Tuple[float,float]]:
        """중심선 양끝점 (start, end). start는 건물 중심 c, end는 c + L·d와 거의 일치."""
        hx = 0.5 * self.length * self.dx
        hy = 0.5 * self.length * self.dy
        # 중심선 = [center - (L/2)d, center + (L/2)d]
        return (self.cx - hx, self.cy - hy), (self.cx + hx, self.cy + hy)

    def corners(self) -> List[Tuple[float, float]]:
        """
        직사각형 꼭짓점 4개를 시계방향으로 반환.
        길이 방향 단위벡터 = (dx,dy), 폭 방향 법선 = (-dy, dx).
        """
        hx = 0.5 * self.length * self.dx
        hy = 0.5 * self.length * self.dy
        nx, ny = -self.dy, self.dx
        half_w = 0.5 * self.width

        c1x, c1y = self.cx - hx, self.cy - hy   # 중심선 시작(건물 중심 쪽)
        c2x, c2y = self.cx + hx, self.cy + hy   # 중심선 끝(그림자 끝)

        p1 = (c1x - half_w*nx, c1y - half_w*ny)
        p2 = (c1x + half_w*nx, c1y + half_w*ny)
        p3 = (c2x + half_w*nx, c2y + half_w*ny)
        p4 = (c2x - half_w*nx, c2y - half_w*ny)
        return [p1, p2, p3, p4]


In [12]:
ARTIFACTS_DIR = Path("../artifacts")

# 그래프 (ENU 좌표 포함)
with open(ARTIFACTS_DIR / "graph_xy.pkl", "rb") as f:
    G = pickle.load(f)

# 그림자 직사각형들
with open(ARTIFACTS_DIR / "shadow_rects.pkl", "rb") as f:
    shadow_rects = pickle.load(f)

# 태양정보 포함된 건물 데이터 (선택 확인용)
b_with_sun = pd.read_parquet(ARTIFACTS_DIR / "buildings_with_sun.parquet")

print("로드 완료:", G.number_of_nodes(), "nodes /", G.number_of_edges(), "edges /", len(shadow_rects), "rectangles")

로드 완료: 330 nodes / 440 edges / 87 rectangles


In [13]:
import math
from statistics import mean

# 지도 중심을 그래프 노드 평균으로 잡자(기준점)
lat0 = mean([G.nodes[n]["lat"] for n in G.nodes])
lng0 = mean([G.nodes[n]["lng"] for n in G.nodes])
R = 6371000.0  # Earth radius (m)

def xy_to_ll(x: float, y: float, lat0: float = lat0, lng0: float = lng0):
    """로컬 평면(ENU) x,y(m) -> 위경도(lat,lng)로 역변환"""
    dlat = y / R
    dlng = x / (R * math.cos(math.radians(lat0)))
    lat = math.degrees(dlat) + lat0
    lng = math.degrees(dlng) + lng0
    return lat, lng


In [14]:
import folium

m = folium.Map(location=[lat0, lng0], zoom_start=16, tiles="cartodbpositron")

# 전체 간선
for u, v in G.edges:
    folium.PolyLine(
        [(G.nodes[u]["lat"], G.nodes[u]["lng"]),
         (G.nodes[v]["lat"], G.nodes[v]["lng"])],
        weight=2, opacity=0.35, color="#3388ff"
    ).add_to(m)

# 출발/도착 마커 (아이콘/색 지정)
folium.Marker(
    path_latlngs[0],
    icon=folium.Icon(color="green", icon="play", prefix="fa"),
    popup=f"START: {path[0]}"
).add_to(m)
folium.Marker(
    path_latlngs[-1],
    icon=folium.Icon(color="red", icon="flag-checkered", prefix="fa"),
    popup=f"END: {path[-1]}"
).add_to(m)


# 최단 경로 (path, info는 Step 7에서 얻은 결과)
path_latlngs = [(G.nodes[n]["lat"], G.nodes[n]["lng"]) for n in path]
folium.PolyLine(path_latlngs, weight=6, color="#ff4d4f", opacity=0.9).add_to(m)


In [15]:
def add_shadow_rects_to_map(m, rects, max_items=None, name="Shadows", fill=True):
    """shadow_rects를 folium Polygon으로 올리기"""
    fg = folium.FeatureGroup(name=name, show=True)
    count = 0
    for r in rects:
        if (max_items is not None) and (count >= max_items):
            break
        pts_xy = r.corners()                         # [(x,y), (x,y), (x,y), (x,y)]
        pts_ll = [xy_to_ll(x, y) for x, y in pts_xy] # → [(lat,lng),...]

        folium.Polygon(
            locations=pts_ll,
            color="#000000",
            weight=1,
            fill=fill,
            fill_opacity=0.20 if fill else 0.0
        ).add_to(fg)
        count += 1
    fg.add_to(m)

# 예) 전부 올리면 무거울 수 있으니 우선 400개만
add_shadow_rects_to_map(m, shadow_rects, max_items=400, name="Shadows", fill=True)

# 레이어 토글
folium.LayerControl(collapsed=False).add_to(m)


In [16]:
m.save("../artifacts/route_map_with_shadows.html")